# 03-8. ベイズ推論 — 動かして確かめる

📖 解説: [`../08_bayesian_inference.md`](../08_bayesian_inference.md)

上から順に **Shift+Enter** で実行していってください。

## このノートで触るもの
1. 事前 × 尤度 → 事後 を目で見る
2. 【対話】事前分布と観測データをスライダーで動かす
3. 共役更新 3 種 (Bernoulli–Beta / Poisson–Gamma / 正規–正規)
4. 事前分布への感度 — 集中度 $\kappa_0$ は「仮想的なデータ件数」
5. 点推定 3 種 (MLE / 事後平均 / MAP) はどこの点か
6. 等裾信用区間 vs HPD 区間 — 歪んだ分布で何が違うか
7. 【対話】損失から決めるロールアウト判断
8. JAX: `jax.grad` で MAP を求める

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (08_bayesian_inference.md)](../08_bayesian_inference.md)

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)
from ipywidgets import interact, IntSlider, FloatSlider

%matplotlib inline

rng = np.random.default_rng(42)

## 1. 事前 × 尤度 → 事後 を目で見る

$$
\pi(\theta \mid x) \propto L(\theta; x)\,\pi(\theta)
$$

「事前の山」に「尤度の山」を掛けると「事後の山」になる。それだけの話です。

設定: 事前 $\mathrm{Beta}(2,2)$、データ「20 人中 12 人成約」→ 事後 $\mathrm{Beta}(14,10)$。

In [ ]:
ALPHA_0: float = 2.0   # 事前 Beta(α₀, β₀)
BETA_0: float = 2.0
N: int = 20            # 試行数 (人)
Y: int = 12            # 成功数 (人)

theta = np.linspace(0.001, 0.999, 500)          # shape: (500,)

prior = stats.beta.pdf(theta, ALPHA_0, BETA_0)                 # 事前 π(θ)
likelihood = theta**Y * (1 - theta)**(N - Y)                   # 尤度 L(θ;x)
likelihood_scaled = likelihood / np.trapezoid(likelihood, theta)  # 見やすさのため面積 1 に
posterior = stats.beta.pdf(theta, ALPHA_0 + Y, BETA_0 + N - Y)  # 事後 π(θ|x)

plt.figure(figsize=(9, 4.5))
plt.plot(theta, prior, label=f'事前 π(θ) = Beta({ALPHA_0:.0f},{BETA_0:.0f})', lw=2)
plt.plot(theta, likelihood_scaled, label='尤度 L(θ;x) (面積1に規格化)', lw=2, ls=':')
plt.plot(theta, posterior, label=f'事後 π(θ|x) = Beta({ALPHA_0+Y:.0f},{BETA_0+N-Y:.0f})', lw=2.5)
plt.axvline(Y / N, color='gray', ls='--', lw=1, label=f'標本比率 y/n = {Y/N:.2f}')
plt.xlabel('成約率 θ'); plt.ylabel('密度')
plt.title('事後 ∝ 尤度 × 事前')
plt.legend(); plt.tight_layout(); plt.show()

print(f'事前平均 = {ALPHA_0/(ALPHA_0+BETA_0):.3f}')
print(f'標本比率 = {Y/N:.3f}')
print(f'事後平均 = {(ALPHA_0+Y)/(ALPHA_0+BETA_0+N):.3f}  ← 事前と標本の「間」に来る')

### 読み取り

事後の山は、**事前の山と尤度の山の「間」**に来ています。
どちらに近いかは、**事前の自信の強さ ($\kappa_0 = \alpha_0 + \beta_0$) とデータ量 $n$ の比**で決まります。

## 2. 【対話】事前分布と観測データを動かしてみる

スライダーで $\alpha_0, \beta_0, n, y$ を動かして、事後がどう動くか体感してください。

**試してほしいこと**
- $\alpha_0 = \beta_0 = 40$（集中度 80 の強い事前）にして、$n = 20$ のデータで動くか
- $n$ を 200 まで増やすと、事前の影響がどうなるか

In [ ]:
def show_update(a0: float = 2.0, b0: float = 2.0, n: int = 20, y: int = 12) -> None:
    """事前・尤度・事後を重ねて描く。

    Args:
        a0: 事前 Beta の成功側パラメータ。
        b0: 事前 Beta の失敗側パラメータ。
        n: 試行数 (人)。
        y: 成功数 (人)。y > n のときは n に丸める。
    """
    y = min(y, n)
    th = np.linspace(0.001, 0.999, 500)
    post_a, post_b = a0 + y, b0 + n - y

    plt.figure(figsize=(9, 4))
    plt.plot(th, stats.beta.pdf(th, a0, b0), label=f'事前 Beta({a0:.0f},{b0:.0f})', lw=2)
    plt.plot(th, stats.beta.pdf(th, post_a, post_b),
             label=f'事後 Beta({post_a:.0f},{post_b:.0f})', lw=2.5)
    plt.axvline(y / n, color='gray', ls='--', lw=1, label=f'標本比率 {y/n:.2f}')
    plt.xlabel('成約率 θ'); plt.ylabel('密度'); plt.ylim(0, 12)
    plt.title(f'集中度 κ₀ = {a0+b0:.0f} (= 仮想的なデータ件数)  vs  実データ n = {n}')
    plt.legend(loc='upper left'); plt.tight_layout(); plt.show()

    post = stats.beta(post_a, post_b)
    lo, hi = post.ppf([0.025, 0.975])
    print(f'事後平均 = {post.mean():.3f}   95%等裾信用区間 = [{lo:.3f}, {hi:.3f}]')
    print(f'P(θ > 0.5 | x) = {1 - post.cdf(0.5):.3f}')


interact(show_update,
         a0=FloatSlider(value=2, min=0.5, max=60, step=0.5, description='α₀'),
         b0=FloatSlider(value=2, min=0.5, max=60, step=0.5, description='β₀'),
         n=IntSlider(value=20, min=1, max=200, step=1, description='n (人)'),
         y=IntSlider(value=12, min=0, max=200, step=1, description='y (成約)'));

## 3. 共役更新 3 種 — 「同じことを 3 回やる」

| 知りたい量 | 観測モデル | 事前 | 事後 |
|---|---|---|---|
| 成功確率 $\theta$ | Bernoulli／二項 | $\mathrm{Beta}(\alpha_0,\beta_0)$ | $\mathrm{Beta}(\alpha_0+y,\ \beta_0+n-y)$ |
| 発生率 $\lambda$ | Poisson | $\mathrm{Gamma}(a_0,b_0)$ | $\mathrm{Gamma}(a_0+\sum x_i,\ b_0+n)$ |
| 平均 $\mu$ | 正規 ($\sigma^2$ 既知) | $N(\mu_0,\tau_0^2)$ | $N(\mu_n,\tau_n^2)$ |

違うのは**分布名だけ**で、手順は完全に同じです。

In [ ]:
# === ① Bernoulli–Beta: 成功側に y、失敗側に n−y を足すだけ ===
alpha_post: float = ALPHA_0 + Y            # 14
beta_post: float = BETA_0 + (N - Y)        # 10
post_beta = stats.beta(alpha_post, beta_post)

print('--- ① Bernoulli–Beta ---')
print(f'事後 Beta({alpha_post:.0f}, {beta_post:.0f}),  事後平均 = {post_beta.mean():.4f}')

# 事後予測: 次の m 人のうち何人成約するか -> ベータ二項分布
M: int = 10
pred_bb = stats.betabinom(M, alpha_post, beta_post)
print(f'次の{M}人の成約数: 期待値 {pred_bb.mean():.2f} 人, 分散 {pred_bb.var():.2f}')

# ⚠️ 「θ を 0.583 に決め打ち」した二項分布と比べると、分散が大きい (過分散)
pred_binom = stats.binom(M, post_beta.mean())
print(f'θ を決め打ちした二項分布: 期待値 {pred_binom.mean():.2f} 人, 分散 {pred_binom.var():.2f}')
print('  -> θ の不確実性を含めた分だけ、ベータ二項のほうが分散が大きい')

In [ ]:
# === ② Poisson–Gamma: 合計件数を形状に、日数を率に足す ===
A_0: float = 2.0     # 事前 Gamma(a₀, b₀) の形状パラメータ
B_0: float = 1.0     # 率パラメータ (rate)
x_counts = np.array([1, 3, 2, 4, 2])       # shape: (5,) 5日間の件数 (件/日)

a_post: float = A_0 + x_counts.sum()       # 14
b_post: float = B_0 + x_counts.size        # 6

# ⚠️ SciPy は scale 流儀なので scale = 1 / rate (最頻出の事故ポイント)
post_gamma = stats.gamma(a=a_post, scale=1.0 / b_post)

print('--- ② Poisson–Gamma ---')
print(f'事後 Gamma({a_post:.0f}, {b_post:.0f}) ※第2引数は率')
print(f'事後平均 E[λ|x] = {post_gamma.mean():.4f} 件/日  (= 14/6)')

# 事後予測: 翌日の件数 -> 負の二項分布
pred_nb = stats.nbinom(n=a_post, p=b_post / (b_post + 1))
print(f'翌日の件数: 期待値 {pred_nb.mean():.3f} 件, 分散 {pred_nb.var():.3f}')
print(f'  ※ Poisson なら 平均 = 分散 = {post_gamma.mean():.3f} のはず -> 負の二項は過分散')

In [ ]:
# === ③ 正規–正規: 事後平均は「事前平均と標本平均の加重平均」 ===
SIGMA2: float = 4.0     # σ² 観測のばらつき (既知、単位: 個²)
MU_0: float = 10.0      # μ₀ 事前平均 (単位: 個)
TAU0_2: float = 1.0     # τ₀² 事前分散

x_demand = np.array([11.0, 12.0, 13.0, 12.0])   # shape: (4,) 需要 (個)
n_d: int = x_demand.size
x_bar: float = float(x_demand.mean())

tau_n2: float = 1.0 / (1.0 / TAU0_2 + n_d / SIGMA2)
mu_n: float = tau_n2 * (MU_0 / TAU0_2 + n_d * x_bar / SIGMA2)

print('--- ③ 正規–正規 ---')
print(f'τn² = {tau_n2:.4f},  μn = {mu_n:.4f}')

# 加重平均の形でも同じ値になることを確認
w_prior: float = SIGMA2 / (SIGMA2 + n_d * TAU0_2)
mu_n_alt: float = w_prior * MU_0 + (1 - w_prior) * x_bar
print(f'加重平均で再計算 = {mu_n_alt:.4f}  (事前の重み {w_prior:.2f} : データの重み {1-w_prior:.2f})')
assert np.isclose(mu_n, mu_n_alt)

# 事後予測の分散は σ² + τn² (観測のばらつき + μ に残る不確実性)
post_norm = stats.norm(mu_n, np.sqrt(tau_n2))
pred_norm = stats.norm(mu_n, np.sqrt(SIGMA2 + tau_n2))
print(f'μ の 95% 信用区間    : [{post_norm.ppf(0.025):.2f}, {post_norm.ppf(0.975):.2f}]')
print(f'翌日需要の 95% 予測区間: [{pred_norm.ppf(0.025):.2f}, {pred_norm.ppf(0.975):.2f}]')
print(f'  ※ 予測区間のほうが広い (分散 {SIGMA2:.1f} + {tau_n2:.1f} = {SIGMA2+tau_n2:.1f})')

## 4. 事前分布への感度 — 集中度 $\kappa_0$ は「仮想的なデータ件数」

**同じデータ（12 勝 8 敗）**に対して、事前分布だけを変えます。

$\kappa_0 = \alpha_0 + \beta_0$ は「すでに何件見たのと同じ重みか」を表します。
$\mathrm{Beta}(40,60)$ は $\kappa_0 = 100$ なので、**20 件の実データに勝ってしまいます**。

In [ ]:
PRIORS = [(1.0, 1.0), (2.0, 2.0), (8.0, 12.0), (40.0, 60.0)]
GAMMA: float = 0.05     # 区間から外す確率 (95% 区間)

print(f'{"事前":>14} {"事前平均":>8} {"κ₀":>5} {"事後":>16} {"事後平均":>8}  95%等裾信用区間')
print('-' * 78)
plt.figure(figsize=(9, 4.5))
th = np.linspace(0.001, 0.999, 500)
for a0, b0 in PRIORS:
    a1, b1 = a0 + Y, b0 + (N - Y)
    post = stats.beta(a1, b1)
    lo, hi = post.ppf([GAMMA / 2, 1 - GAMMA / 2])
    print(f'  Beta({a0:>4.0f},{b0:>4.0f}) {a0/(a0+b0):>8.2f} {a0+b0:>5.0f}'
          f'   Beta({a1:>3.0f},{b1:>3.0f}) {post.mean():>8.3f}  [{lo:.3f}, {hi:.3f}]')
    plt.plot(th, post.pdf(th), lw=2, label=f'事前 Beta({a0:.0f},{b0:.0f}) → 事後平均 {post.mean():.3f}')

plt.axvline(Y / N, color='gray', ls='--', lw=1, label=f'標本比率 {Y/N:.2f}')
plt.xlabel('成約率 θ'); plt.ylabel('密度')
plt.title('同じデータ (12勝8敗) でも、事前分布で結論は動く')
plt.legend(); plt.tight_layout(); plt.show()

### 読み取り

- 弱い事前分布どうし（$\mathrm{Beta}(1,1)$ と $\mathrm{Beta}(2,2)$）で結果が近ければ、結論は比較的**頑健**
- 強い事前分布で結論が変わるなら、その依存性を**感度分析として報告する**

> 「ベイズは主観的でズルい」への回答は、**主観を隠さず明示して、依存性を報告すること**です。

## 5. 点推定 3 種 — どこの点を見ているか

| 推定量 | 式 | 何の点か |
|---|---|---|
| MLE | $y/n$ | 尤度の頂点（事前を使わない） |
| 事後平均 | $\dfrac{\alpha}{\alpha+\beta}$ | 事後分布の**重心** |
| MAP | $\dfrac{\alpha-1}{\alpha+\beta-2}$ | 事後密度の**最大点**（山の頂上） |

In [ ]:
mle: float = Y / N
post_mean: float = alpha_post / (alpha_post + beta_post)
map_est: float = (alpha_post - 1) / (alpha_post + beta_post - 2)

print(f'MLE      = {mle:.4f}   (尤度の頂点)')
print(f'事後平均 = {post_mean:.4f}   (事後分布の重心)')
print(f'MAP      = {map_est:.4f}   (事後密度の最大点)')

th = np.linspace(0.001, 0.999, 800)
pdf = post_beta.pdf(th)

plt.figure(figsize=(9, 4))
plt.plot(th, pdf, lw=2.5, color='C0', label=f'事後 Beta({alpha_post:.0f},{beta_post:.0f})')
plt.fill_between(th, pdf, alpha=0.15)
for value, name, color in [(mle, 'MLE', 'C3'), (post_mean, '事後平均', 'C2'), (map_est, 'MAP', 'C1')]:
    plt.axvline(value, color=color, ls='--', lw=2, label=f'{name} = {value:.3f}')
plt.xlim(0.2, 0.95); plt.xlabel('成約率 θ'); plt.ylabel('密度')
plt.title('点推定は「事後分布の要約」にすぎない')
plt.legend(); plt.tight_layout(); plt.show()

print('\n※ この例では 3 つとも近いが、歪んだ分布では重心 (事後平均) と頂上 (MAP) は大きくズレる')

## 6. 等裾信用区間 vs HPD 区間

- **等裾**: 左右の裾に $\gamma/2$ ずつ残す。$P(\theta < L \mid x) = P(\theta > U \mid x) = \gamma/2$
- **HPD**: 密度の高いところから $1-\gamma$ を集める。$C(c) = \{\theta : \pi(\theta \mid x) \ge c\}$

**対称な単峰分布では一致**します。歪んでいると差が出るので、右に裾を引く $\mathrm{Beta}(2,8)$ で比べます。

In [ ]:
from scipy.optimize import minimize_scalar


def hpd_interval(dist, cred: float = 0.95) -> tuple[float, float]:
    """連続分布の HPD 区間を数値的に求める。

    「幅が最短になる区間」を探すことで、密度の高い領域を優先する区間を得る。

    ⚠️ 単峰 (unimodal) の分布を前提にした実装。山が 2 つある分布では
    HPD 領域は離れた複数の区間の和になるため、この関数では扱えない。

    Args:
        dist: scipy の凍結分布 (frozen distribution)。
        cred: 含めたい確率 (1 − γ)。

    Returns:
        (下限, 上限) のタプル。
    """
    def width(lo: float) -> float:
        return float(dist.ppf(dist.cdf(lo) + cred) - lo)

    upper_bound = float(dist.ppf(1 - cred))
    res = minimize_scalar(width, bounds=(float(dist.ppf(1e-6)), upper_bound - 1e-9),
                          method='bounded')
    lo = float(res.x)
    return lo, float(dist.ppf(dist.cdf(lo) + cred))


CRED: float = 0.95
for a, b in [(2.0, 8.0), (14.0, 10.0)]:
    d = stats.beta(a, b)
    et_lo, et_hi = d.ppf([(1 - CRED) / 2, 1 - (1 - CRED) / 2])
    hpd_lo, hpd_hi = hpd_interval(d, CRED)
    print(f'Beta({a:.0f},{b:.0f}):')
    print(f'  等裾 {CRED:.0%} = [{et_lo:.4f}, {et_hi:.4f}]  幅 {et_hi-et_lo:.4f}')
    print(f'  HPD  {CRED:.0%} = [{hpd_lo:.4f}, {hpd_hi:.4f}]  幅 {hpd_hi-hpd_lo:.4f}  ← 常に最短')

# 歪んだ分布で図示
d = stats.beta(2, 8)
th = np.linspace(0.0001, 0.7, 800)
et_lo, et_hi = d.ppf([0.025, 0.975])
hpd_lo, hpd_hi = hpd_interval(d, CRED)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (lo, hi, title) in zip(axes, [(et_lo, et_hi, '等裾 95%'), (hpd_lo, hpd_hi, 'HPD 95%')]):
    ax.plot(th, d.pdf(th), lw=2)
    mask = (th >= lo) & (th <= hi)
    ax.fill_between(th[mask], d.pdf(th[mask]), alpha=0.35)
    ax.set_title(f'{title}: [{lo:.3f}, {hi:.3f}] 幅 {hi-lo:.3f}')
    ax.set_xlabel('θ')
axes[0].set_ylabel('密度')
fig.suptitle('右に裾を引く Beta(2,8) — 等裾は密度の低い側まで拾ってしまう')
plt.tight_layout(); plt.show()

### ⚠️ 信用区間と信頼区間は「読み方」が違う

| | ベイズ**信用**区間 | 頻度論的**信頼**区間 |
|---|---|---|
| 確率が乗る対象 | 未知パラメータ $\theta$ | 標本ごとに変わる**区間のほう** |
| 読み方 | 「$\theta$ がこの区間に入る確率が 95%」と**言ってよい** | 「手続きを繰り返すと 95% が真値を含む」 |

[`06_estimation.md`](../06_estimation.md) で「信頼区間を『真値が 95% の確率で入る』と読むのは誤り」と学びました。
その **"正しく読める版"** がベイズの信用区間です。

## 7. 【対話】損失から決めるロールアウト判断

$$
P(\theta_{\mathrm{new}} > \theta_{\mathrm{old}} \mid x) > \frac{C_{10}}{C_{10} + C_{01}}
$$

- $C_{10}$ = ダメなのに導入した損失
- $C_{01}$ = 良いのに見送った機会損失

スライダーで損失を動かして、**しきい値がどう動くか**を見てください。

In [ ]:
def rollout_decision(c10: float = 100.0, c01: float = 50.0,
                     n_new: int = 20, y_new: int = 12) -> None:
    """新旧 UI の事後分布を比べ、損失に基づくロールアウト判断を表示する。

    Args:
        c10: 誤って導入したときの損失 (共通尺度、例: 万円)。
        c01: 良いのに見送ったときの機会損失。
        n_new: 新 UI の表示人数 (人)。
        y_new: 新 UI の成約数 (人)。
    """
    y_new = min(y_new, n_new)
    # 一様事前 Beta(1,1) から更新
    post_new = stats.beta(1 + y_new, 1 + n_new - y_new)
    post_old = stats.beta(1 + 90, 1 + 110)      # 旧 UI: 200 人中 90 人成約 (固定)

    K: int = 100_000                             # モンテカルロの標本数
    mc = np.random.default_rng(0)
    s_new = post_new.rvs(K, random_state=mc)     # shape: (100000,)
    s_old = post_old.rvs(K, random_state=mc)
    p_better = float((s_new > s_old).mean())
    threshold = c10 / (c10 + c01)

    th = np.linspace(0.001, 0.999, 400)
    plt.figure(figsize=(9, 3.6))
    plt.plot(th, post_old.pdf(th), lw=2, label='旧UI 事後 (90/200)')
    plt.plot(th, post_new.pdf(th), lw=2, label=f'新UI 事後 ({y_new}/{n_new})')
    plt.xlabel('成約率 θ'); plt.ylabel('密度'); plt.legend(); plt.tight_layout(); plt.show()

    verdict = '✅ ロールアウトする' if p_better > threshold else '⛔ 見送る'
    print(f'P(θ_new > θ_old | x) = {p_better:.3f}')
    print(f'必要な確信度 C₁₀/(C₁₀+C₀₁) = {threshold:.3f}')
    print(f'-> {verdict}')


interact(rollout_decision,
         c10=FloatSlider(value=100, min=10, max=500, step=10, description='C₁₀ 誤導入'),
         c01=FloatSlider(value=50, min=10, max=500, step=10, description='C₀₁ 見送り'),
         n_new=IntSlider(value=20, min=5, max=300, step=5, description='新UI n'),
         y_new=IntSlider(value=12, min=0, max=300, step=1, description='新UI y'));

### 読み取り

- **誤導入の損失 $C_{10}$ が大きい → しきい値が上がる**（慎重になる）
- **見送りの損失 $C_{01}$ が大きい → しきい値が下がる**（低い確率でも踏み込む）

「5% 有意水準を全部の場面で使う」運用との一番の実務差がここです。
**しきい値を業務の損失から決められる。**

## 8. JAX: `jax.grad` で MAP を求める

共役モデルは代数で解けるので JAX の出番は薄いのですが、
「**対数事後密度を書いて、勾配で山頂を探す**」形は非共役モデル（[`../09_mcmc.md`](../09_mcmc.md)）にそのまま繋がります。

In [ ]:
# === 標準形式 (閉じた式) ===
map_closed: float = (alpha_post - 1) / (alpha_post + beta_post - 2)
print(f'MAP (閉じた式) = {map_closed:.6f}')

# === JAX 形式 (jax.grad で勾配上昇) ===
import jax
import jax.numpy as jnp

A0_J, B0_J, N_J, Y_J = 2.0, 2.0, 20.0, 12.0


def log_posterior(theta_j: jnp.ndarray) -> jnp.ndarray:
    """正規化定数を除いた対数事後密度 log π(θ|x) + const。

    Args:
        theta_j: 成約率 (スカラー)。

    Returns:
        対数事後密度 (スカラー)。定義域外は -inf。
    """
    inside = (theta_j > 0) & (theta_j < 1)
    t = jnp.clip(theta_j, 1e-12, 1 - 1e-12)         # log(0) 回避
    lp = (A0_J + Y_J - 1) * jnp.log(t) + (B0_J + N_J - Y_J - 1) * jnp.log(1 - t)
    return jnp.where(inside, lp, -jnp.inf)


grad_log_post = jax.grad(log_posterior)   # 手で微分しない

LEARNING_RATE: float = 1e-3
N_STEPS: int = 500
theta_j = jnp.float32(0.5)
history = []
for _ in range(N_STEPS):
    theta_j = theta_j + LEARNING_RATE * grad_log_post(theta_j)   # 勾配"上昇"
    history.append(float(theta_j))

print(f'MAP (jax.grad) = {float(theta_j):.6f}')

# --- 検算: 標準形式と JAX 形式が一致するか ---
assert abs(float(theta_j) - map_closed) < 1e-4
print('✅ 標準形式と JAX 形式が一致')

plt.figure(figsize=(8, 3.2))
plt.plot(history, lw=2)
plt.axhline(map_closed, color='C3', ls='--', label=f'閉じた式の MAP = {map_closed:.4f}')
plt.xlabel('勾配上昇のステップ'); plt.ylabel('θ')
plt.title('jax.grad による MAP 探索')
plt.legend(); plt.tight_layout(); plt.show()

## まとめ

- **事後 ∝ 尤度 × 事前**。事後の山は事前と尤度の「間」に来る
- **集中度 $\kappa_0 = \alpha_0+\beta_0$ は仮想的なデータ件数**。強い事前は実データに勝ってしまう
- **共役モデル 3 種は同じ手順**。違うのは分布名だけ
- **事後予測分布は $\theta$ を決め打ちしない**ので、二項ではなくベータ二項、Poisson ではなく負の二項になる（過分散）
- **点推定は事後分布の要約**。MLE / 事後平均 / MAP は見ている点が違う
- **等裾と HPD** は歪んだ分布でズレる。HPD は常に最短
- **信用区間は $\theta$ に確率が乗る**。信頼区間とは読み方が違う
- **しきい値は損失から決められる** — これがベイズの実務的な強み

この章の核心:

> **ベイズの成果物は事後分布ひとつ。**
> **点推定・区間・予測・意思決定は、すべてそこからの「取り出し方」の違いにすぎない。**

→ 次は [`09_mcmc.ipynb`](09_mcmc.ipynb) — 事後分布に名前が付かないとき、どうやって計算するか

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [`07_hypothesis_testing.ipynb`](07_hypothesis_testing.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`09_mcmc.ipynb`](09_mcmc.ipynb) |